In [1]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from sklearn.metrics import jaccard_score
import matplotlib.pyplot as plt

d:\PROJ\vesselmonitoring\ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 5
LR = 3e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_DIR = "data/train/images/"
MASK_DIR  = "data/train/labels/"

In [4]:
COLOR_MAPPING = {
    (0, 0, 0): 0,        # Sea
    (0, 255, 255): 1,    # Oil
    (255, 0, 0): 2,      # Look-alikes
    (153, 76, 0): 3,     # Ships
    (0, 153, 0): 4       # Land
}

def rgb_to_class(mask):
    mask = mask.astype(np.uint8)
    class_mask = np.zeros(mask.shape[:2], dtype=np.int64)

    for rgb, cls in COLOR_MAPPING.items():
        matches = np.all(mask == rgb, axis=-1)
        class_mask[matches] = cls

    return class_mask

In [5]:
def get_train_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),

        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5),

        A.RandomBrightnessContrast(p=0.3),
        A.HueSaturationValue(p=0.3),

        A.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)),
        ToTensorV2()
    ])

In [6]:
class OilDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transforms=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transforms = transforms

        self.images = sorted(os.listdir(img_dir))
        self.masks  = sorted(os.listdir(mask_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = np.array(Image.open(os.path.join(self.img_dir, self.images[idx])).convert("RGB"))
        mask = np.array(Image.open(os.path.join(self.mask_dir, self.masks[idx])).convert("RGB"))

        mask = rgb_to_class(mask)

        if self.transforms:
            aug = self.transforms(image=img, mask=mask)
            img = aug["image"]
            mask = aug["mask"]

        return img, mask.long()

In [7]:
full_dataset = OilDataset(IMAGE_DIR, MASK_DIR, transforms=None)

train_size = int(0.7 * len(full_dataset))
val_size   = int(0.15 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

# Assign transforms AFTER split
train_ds.dataset.transforms = get_train_transforms()
val_ds.dataset.transforms   = get_val_transforms()
test_ds.dataset.transforms  = get_val_transforms()

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [8]:
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b5",
    encoder_weights="imagenet",
    classes=5,
    activation=None
).to(DEVICE)

In [9]:
dice_loss = smp.losses.DiceLoss(mode="multiclass")
focal_loss = smp.losses.FocalLoss(mode="multiclass")

def loss_fn(pred, target):
    return dice_loss(pred, target) + focal_loss(pred, target)

In [10]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

In [11]:
def train_epoch(loader):
    model.train()
    total_loss = 0

    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        preds = model(imgs)

        loss = loss_fn(preds, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def val_epoch(loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

            preds = model(imgs)
            loss = loss_fn(preds, masks)

            total_loss += loss.item()

    return total_loss / len(loader)

In [12]:
best_val = float("inf")

for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader)
    val_loss = val_epoch(val_loader)

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}: Train {train_loss:.4f} | Val {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_model.pth")

Epoch 1: Train 1.3353 | Val 0.9734
Epoch 2: Train 0.8269 | Val 0.7825
Epoch 3: Train 0.7109 | Val 0.6939
Epoch 4: Train 0.6379 | Val 0.8001
Epoch 5: Train 0.5780 | Val 0.5507


In [13]:
def evaluate(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(DEVICE)

            preds = torch.argmax(model(imgs), dim=1).cpu().numpy()
            labels = masks.numpy()

            preds_all.extend(preds.flatten())
            labels_all.extend(labels.flatten())

    iou = jaccard_score(labels_all, preds_all, average='macro')
    print("Mean IoU:", iou)

evaluate(test_loader)

Mean IoU: 0.5944407377308245


In [ ]:
def visualize(img, pred):
    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.imshow(img)
    plt.title("Image")

    plt.subplot(1,2,2)
    plt.imshow(pred, cmap='jet')
    plt.title("Prediction")

    plt.show()